**Project 3**

**Submitted by:** Banu Boopalan

**Date:** 4/8/2025

**Course:** Data Science – DATA620

**Video Link:** 

In [21]:
# Import necessary libraries
import nltk
from nltk.corpus import names
import random
from sklearn.metrics import classification_report, accuracy_score

nltk.download('names')

# Prepare labeled names dataset
labeled_names = [(name, 'male') for name in names.words('male.txt')] + \
             [(name, 'female') for name in names.words('female.txt')]

random.seed(42)
random.shuffle(labeled_names)
train_set = labeled_names[1000:]
devtest_set = labeled_names[500:1000]
test_set = labeled_names[:500]

[nltk_data] Downloading package names to
[nltk_data]     C:\Users\Banu\AppData\Roaming\nltk_data...
[nltk_data]   Package names is already up-to-date!


Use the first round of baseline gender features using last tetter, last 2 letters, first letter, total num of characters and vowel count. Train basic classifiers NB, DT, and Maxentclassifer.

The baseline maximum entropy model has Based on the provided performance metrics, gives an accuracy of **0.778**, and the Naive Bayes (0.798). Decision tree has  (0.758).  On the **Test Set**, MaxEnt gets **0.784**, and better than both NB (0.776) and Decision Tree (0.732). The Maxmimum entropy model uses logistic regression to give weights to features and if they are correlated, it models the interactions better than the naive bayes which requires features be independence but entropy model does not require that. Compared to decision trees the entropy model does not overfit as much. 


In [22]:
# Define feature extraction function
def gender_features(name):
    return {
        'last_letter': name[-1].lower(),
        'last_two_letters': name[-2:].lower(),
        'first_letter': name[0].lower(),
        'length': len(name),
        'vowel_count': sum(1 for char in name.lower() if char in 'aeiou'),
    }

# Prepare feature sets
train_features = [(gender_features(name), gender) for name, gender in train_set]
devtest_features = [(gender_features(name), gender) for name, gender in devtest_set]
test_features = [(gender_features(name), gender) for name, gender in test_set]

# Train classifiers
classifier_nb = nltk.NaiveBayesClassifier.train(train_features)
classifier_dt = nltk.DecisionTreeClassifier.train(train_features)
classifier_me = nltk.classify.MaxentClassifier.train(train_features, max_iter=10)

# Evaluate classifiers
def evaluate_classifier(classifier, name):
    devtpredictions = [classifier.classify(features) for features, _ in devtest_features]
    devtlabels = [label for _, label in devtest_features]
    print(f"\n{name} - Dev-Test Set Performance:")
    print(classification_report(devtlabels, devtpredictions))
    print(f"Accuracy on Dev-Test Set ({name}):", accuracy_score(devtlabels, devtpredictions))

    test_predictions = [classifier.classify(features) for features, _ in test_features]
    test_labels = [label for _, label in test_features]
    print(f"\n{name} - Test Set Performance:")
    print(classification_report(test_labels, test_predictions))
    print(f"Accuracy on Test Set ({name}):", accuracy_score(test_labels, test_predictions))

evaluate_classifier(classifier_nb, "Naive Bayes")
evaluate_classifier(classifier_dt, "Decision Tree")
evaluate_classifier(classifier_me, "Maximum Entropy")

# Show most informative features for Naive Bayes
print("\nNaive Bayes - Most Informative Features:")
classifier_nb.show_most_informative_features(10)

  ==> Training (10 iterations)

      Iteration    Log Likelihood    Accuracy
      ---------------------------------------
             1          -0.69315        0.368
             2          -0.44368        0.778
             3          -0.38263        0.806
             4          -0.35296        0.810
             5          -0.33602        0.814
             6          -0.32524        0.816
             7          -0.31783        0.817
             8          -0.31246        0.819
             9          -0.30839        0.821
         Final          -0.30520        0.822

Naive Bayes - Dev-Test Set Performance:
              precision    recall  f1-score   support

      female       0.84      0.81      0.83       297
        male       0.74      0.77      0.76       203

    accuracy                           0.80       500
   macro avg       0.79      0.79      0.79       500
weighted avg       0.80      0.80      0.80       500

Accuracy on Dev-Test Set (Naive Bayes): 0.798

N

Based on the code from book  ("Natural Language Processing in Python") we can look at errors and misclassifications. It looks like female names are being classified as male. 

In [23]:
# Analyze errors
errors = []
for (name, tag) in devtest_set:
    guess = classifier_nb.classify(gender_features(name))
    if guess != tag:
        errors.append((tag, guess, name))

for (tag, guess, name) in sorted(errors):
    print('correct={:<8} guess={:<8s} name={:<30}'.format(tag, guess, name))

correct=female   guess=male     name=Beatriz                       
correct=female   guess=male     name=Beth                          
correct=female   guess=male     name=Bliss                         
correct=female   guess=male     name=Brigid                        
correct=female   guess=male     name=Brooks                        
correct=female   guess=male     name=Charil                        
correct=female   guess=male     name=Dallas                        
correct=female   guess=male     name=Debby                         
correct=female   guess=male     name=Doralin                       
correct=female   guess=male     name=Dory                          
correct=female   guess=male     name=Eden                          
correct=female   guess=male     name=Eleanor                       
correct=female   guess=male     name=Gillan                        
correct=female   guess=male     name=Ginnifer                      
correct=female   guess=male     name=Glad       

The above model was classifying Name "Trinity" as male, therefore we can add trin below as a pattern and ity ending with as a pattern. Additionally we added more variations of the extractions, looking at ration of vowels to consonants , count occurance of a and e and y and us ,like ending in a like Anna are names such as female. 
On this round of model run, the NB classifier worked better and classified Neo as a male and Trinity as female.

Male : Out of all names predicted as "male," 76% are male
Female : Out of all names predicted as "female," 85% are female.
This model predicted  "female" names better since there is a higher precision (how many true positives were predicted) and recall for the female class.


In [24]:


def gender_features3(name):
    return {
        'last_letter': name[-1].lower(),
        'last_two_letters': name[-2:].lower(),
        'last_three_letters': name[-3:].lower() if len(name) > 2 else '',
        'first_letter': name[0].lower(),
        'first_two_letters': name[:2].lower() if len(name) > 1 else '',
        'length': len(name),
        'length_category': 'short' if len(name) <= 4 else 'long' if len(name) > 7 else 'medium',
        'vowel_count': sum(1 for char in name.lower() if char in 'aeiou'),
        'vowel_consonant_ratio': sum(1 for char in name.lower() if char in 'aeiou') / max(1, sum(1 for char in name.lower() if char not in 'aeiou')),
        'contains_y': 'y' in name.lower(),
        'contains_us': 'us' in name.lower(),
        'count_a': name.lower().count('a'),
        'count_e': name.lower().count('e'),
        'contains_trin': 'trin' in name.lower(),
        'ends_with_ity': name.lower().endswith('ity'),
    }

# Train and evaluate classifiers with new features
train_features = [(gender_features3(name), gender) for name, gender in train_set]
devtest_features = [(gender_features3(name), gender) for name, gender in devtest_set]
test_features = [(gender_features3(name), gender) for name, gender in test_set]
classifier_new1 = nltk.NaiveBayesClassifier.train(train_features)
print(nltk.classify.accuracy(classifier_new1, devtest_features))
print(classifier_new1.classify(gender_features3('Trinity')))
print(classifier_new1.classify(gender_features3('Neo')))

errors = []
for (name, tag) in devtest_set:
    guess = classifier_nb.classify(gender_features3(name))
    if guess != tag:
        errors.append((tag, guess, name))

for (tag, guess, name) in sorted(errors):
    print('correct={:<8} guess={:<8s} name={:<30}'.format(tag, guess, name))

evaluate_classifier(classifier_new1, "Naive Bayes")

0.812
female
male
correct=female   guess=male     name=Beatriz                       
correct=female   guess=male     name=Beth                          
correct=female   guess=male     name=Bliss                         
correct=female   guess=male     name=Brigid                        
correct=female   guess=male     name=Brooks                        
correct=female   guess=male     name=Charil                        
correct=female   guess=male     name=Dallas                        
correct=female   guess=male     name=Debby                         
correct=female   guess=male     name=Doralin                       
correct=female   guess=male     name=Dory                          
correct=female   guess=male     name=Eden                          
correct=female   guess=male     name=Eleanor                       
correct=female   guess=male     name=Gillan                        
correct=female   guess=male     name=Ginnifer                      
correct=female   guess=male   

In [ ]:
from sklearn.metrics import confusion_matrix

# True labels and predicted labels
true_labels = [label for _, label in devtest_features]  
predicted_labels = [classifier_new1.classify(features) for features, _ in devtest_features]  # Predicted labels

cm = confusion_matrix(true_labels, predicted_labels, labels=['male', 'female'])

TP = cm[1, 1]  # True Positives (correctly predicted 'female')
TN = cm[0, 0]  # True Negatives (correctly predicted 'male')

print("True Positives (TP):", TP)
print("True Negatives (TN):", TN)

print(classification_report(true_labels, predicted_labels, labels=['male', 'female']))



True Positives (TP): 245
True Negatives (TN): 161
              precision    recall  f1-score   support

        male       0.76      0.79      0.77       203
      female       0.85      0.82      0.84       297

    accuracy                           0.81       500
   macro avg       0.80      0.81      0.81       500
weighted avg       0.81      0.81      0.81       500



The decision tree is lesser in model fit than the Entropy. Decision trees are prone to overfitting, therefore it may have not as good of a performance. Because decision trees don't work with probabilitic output, these may not help when we need probabilities to classify.

There are variations in performance between the dev test set and test set. Decision trees can be biased towards majority class when there is class imbalance

In [26]:
classifier_dt1 = nltk.DecisionTreeClassifier.train(train_features)

errors = []
for (name, tag) in devtest_set:
    guess = classifier_dt1.classify(gender_features3(name))
    if guess != tag:
        errors.append((tag, guess, name))

for (tag, guess, name) in sorted(errors):
    print('correct={:<8} guess={:<8s} name={:<30}'.format(tag, guess, name))
    
evaluate_classifier(classifier_dt1, "DecisionTreeClassifier")

correct=female   guess=male     name=Adorne                        
correct=female   guess=male     name=Brooks                        
correct=female   guess=male     name=Cordy                         
correct=female   guess=male     name=Dallas                        
correct=female   guess=male     name=Dani                          
correct=female   guess=male     name=Dione                         
correct=female   guess=male     name=Dory                          
correct=female   guess=male     name=Eddie                         
correct=female   guess=male     name=Eden                          
correct=female   guess=male     name=Gene                          
correct=female   guess=male     name=Geri                          
correct=female   guess=male     name=Glad                          
correct=female   guess=male     name=Gretchen                      
correct=female   guess=male     name=Grissel                       
correct=female   guess=male     name=Harriott   

The maximum entropy model had the best score - Accuracy on Test Set (Entropy): 0.81

In [27]:
classifier_me1 = nltk.classify.MaxentClassifier.train(train_features, max_iter=10)

errors = []
for (name, tag) in devtest_set:
    guess = classifier_me1.classify(gender_features3(name))
    if guess != tag:
        errors.append((tag, guess, name))

for (tag, guess, name) in sorted(errors):
    print('correct={:<8} guess={:<8s} name={:<30}'.format(tag, guess, name))

evaluate_classifier(classifier_me1, "Entropy")

classifier_me1.show_most_informative_features(20)

  ==> Training (10 iterations)

      Iteration    Log Likelihood    Accuracy
      ---------------------------------------
             1          -0.69315        0.368
             2          -0.51414        0.680
             3          -0.44865        0.800
             4          -0.40463        0.826
             5          -0.37370        0.838
             6          -0.35097        0.843
             7          -0.33358        0.850
             8          -0.31985        0.854
             9          -0.30871        0.858
         Final          -0.29946        0.861
correct=female   guess=male     name=Brigid                        
correct=female   guess=male     name=Brooks                        
correct=female   guess=male     name=Dallas                        
correct=female   guess=male     name=Eden                          
correct=female   guess=male     name=Gillan                        
correct=female   guess=male     name=Ginnifer                      
correct=

This dataset names is imbalanced showing majority class as female (female: 62.95%), therefore the NB and the decision tree have shown lesser performances than the maximum entropy model. 

In [28]:

from collections import Counter

class_counts = Counter([label for _, label in labeled_names])

print("Class Distribution:")
for label, count in class_counts.items():
    print(f"{label}: {count}")

total = sum(class_counts.values())
print("\nClass Percentage Distribution:")
for label, count in class_counts.items():
    print(f"{label}: {count / total * 100:.2f}%")

Class Distribution:
female: 5001
male: 2943

Class Percentage Distribution:
female: 62.95%
male: 37.05%
